# 09b - Does the admission-day note leak information from later in the stay? (run_02)

In the 24-hour version, procedures and medications lost their predictive power, but the admission-day note kept 93%.
Two checks:
1. **Content check**: for each stay, take procedures and medications that STARTED MORE THAN 24 HOURS after admission
   and test whether their names appear in the admission-day note text. Compare with a control: items from a random other
   stay. If later items appear far more often than the control, the note describes the whole stay (leakage).
2. **Section check**: which note section (HPI, allergies, medications, assessment and plan) carries the signal.

Runtime: CPU, about 5 minutes. Needs notebook 07 run twice (whole stay and 24 h).

In [ ]:
# Cell 1 - load
import os, sys, re, json, pickle, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
RAW, W = f"{BASE}/raw/csv", f"{BASE}/features_full_24h"
notes = pickle.load(open(f"{W}/encounter_notes.pkl", "rb"))            # admission-day note, kept sections
X = pd.read_parquet(f"{W}/encounters_524.parquet")
enc = X[["hadm_id", "subject_id", "split", "admittime", "is_plos", "target"]].copy()
cut = enc.set_index("hadm_id")["admittime"] + pd.Timedelta(hours=24)
ids = set(enc.hadm_id)
proc = pd.concat([c[c.ENCOUNTER.isin(ids)] for c in pd.read_csv(f"{RAW}/procedures.csv", usecols=["START", "ENCOUNTER", "DESCRIPTION"], chunksize=200_000)])
med = pd.concat([c[c.ENCOUNTER.isin(ids)] for c in pd.read_csv(f"{RAW}/medications.csv", usecols=["START", "ENCOUNTER", "DESCRIPTION"], chunksize=200_000)])
for d in (proc, med):
    d["START"] = pd.to_datetime(d.START, errors="coerce", utc=True).dt.tz_localize(None)
    d["late"] = d.START > d.ENCOUNTER.map(cut)
    d["name"] = d.DESCRIPTION.str.replace(r"\s*\((procedure|regime/therapy|substance|product)\)", "", regex=True).str.lower().str.strip()
print(f"Stays {len(enc):,} | notes {len(notes):,} | procedure rows {len(proc):,} ({proc.late.mean():.0%} after 24 h) | "
      f"medication rows {len(med):,} ({med.late.mean():.0%} after 24 h)")

In [ ]:
# Cell 2 - content check: do names of LATER items appear in the admission-day note?
rng = np.random.default_rng(0)
text = {h: " ".join(v.values()).lower() for h, v in notes.items()}
hs = list(text)
def hit_rates(items, label):
    rows = []
    late = items[items.late].groupby("ENCOUNTER")["name"].apply(lambda s: sorted(set(s)))
    early = items[~items.late].groupby("ENCOUNTER")["name"].apply(lambda s: sorted(set(s)))
    for h, names in late.items():
        if h not in text or not names: continue
        other = late.iloc[rng.integers(len(late))]           # control: later items of a random other stay
        own_early = early.get(h, [])
        rows.append({"late_in_note": np.mean([n in text[h] for n in names]),
                     "control_in_note": np.mean([n in text[h] for n in other if n not in names]) if [n for n in other if n not in names] else np.nan,
                     "early_in_note": np.mean([n in text[h] for n in own_early]) if len(own_early) else np.nan,
                     "is_plos": enc.set_index("hadm_id").is_plos.get(h, 0)})
    r = pd.DataFrame(rows, columns=["late_in_note", "control_in_note", "early_in_note", "is_plos"])
    if r.empty: return pd.DataFrame(index=[f"{label}: none after 24 h"]), 0
    out = r.groupby("is_plos")[["late_in_note", "early_in_note", "control_in_note"]].mean()
    out.index = [f"{label}: {'PLOS' if i == 1 else 'non-PLOS'}" for i in out.index]
    return out, len(r)
pt, n1 = hit_rates(proc, "procedures"); mt, n2 = hit_rates(med, "medications")
print(f"Stays with later items: procedures {n1:,}, medications {n2:,}")
print("Share of item names found in the admission-day note (late = started after 24 h; control = another stay's items):")
pd.concat([pt, mt]).round(3)

In [ ]:
# Cell 3 - section check: demographics + one note section at a time (same Word2Vec, same classifier)
from gensim.models import Word2Vec
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
w2v = Word2Vec.load(f"{W}/word2vec_notes.model")
def toks(t): return [w for w in re.findall(r"[a-z][a-z_]{2,}", (t or "").lower()) if w in w2v.wv]
def pool(t):
    v = [w2v.wv[w] for w in toks(t)]
    return np.mean(v, 0) if v else np.zeros(w2v.vector_size)
SAFE = ["anchor_age", "gender_enc", "race_enc", "ethnicity_enc"]
res = []
for sec in ["hpi", "allergies", "medications", "assessment_plan", "ALL"]:
    E = np.stack([pool(" ".join(notes.get(h, {}).values()) if sec == "ALL" else notes.get(h, {}).get(sec, "")) for h in X.hadm_id])
    F = pd.concat([X[SAFE].reset_index(drop=True), pd.DataFrame(E)], axis=1); F.columns = F.columns.astype(str)
    tr, te = (X.split == "train").values, (X.split == "test").values
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(F[tr], X.is_plos[tr])
    res.append({"note section": sec, "stays with text": float(np.mean([bool(notes.get(h, {}).get(sec)) for h in X.hadm_id])) if sec != "ALL" else 1.0,
                "AP_test": average_precision_score(X.is_plos[te], m.predict_proba(F[te])[:, 1])})
print(f"Test PLOS prevalence {X.is_plos[X.split == 'test'].mean():.3f}; demographics alone gave about 0.064")
pd.DataFrame(res).set_index("note section").round(4)

In [ ]:
# Cell 4 - read one long PLOS stay: its admission-day note next to what happened later (synthetic data, safe to view)
h = enc[enc.is_plos == 1].sort_values("target").hadm_id.iloc[-len(enc[enc.is_plos == 1]) // 2]
print(f"Stay {h}: length {enc.set_index('hadm_id').target[h]:.0f} days\n")
for k, v in notes.get(h, {}).items(): print(f"--- {k} ---\n{v[:800]}\n")
print("--- procedures after 24 h ---"); print(proc[(proc.ENCOUNTER == h) & proc.late].name.value_counts().head(10).to_string())
print("--- medications after 24 h ---"); print(med[(med.ENCOUNTER == h) & med.late].name.value_counts().head(10).to_string())